# Part 1: reproduce the main result on Colab (LoRA vs LoRA+ on SmolVLM-256M)

This notebook retrains one cell of the main table **from scratch** on a free Colab T4 (Runtime → Change runtime type → T4 GPU).
It runs `part1_loraplus_smolvlm.py --experiment single`, which skips the learning-rate grid and trains one full-budget run
(3000 VQAv2 questions, 2 epochs = 750 steps, fp16, seed 42), then evaluates zero-shot and adapted test accuracy.

| Run | Command flags | Our result (RTX 4090, seed 42) |
|---|---|---|
| Tuned LoRA | `--lr-a 2e-4 --ratio 1` | see `outputs/part1_loraplus/runs/main_LoRA_tuned_lr0.0002_r1_s42` |
| Tuned LoRA+ | `--lr-a 1e-4 --ratio 4` | see `outputs/part1_loraplus/runs/main_LoRAplus_tuned_lr0.0001_r4_s42` |

Expected time on a T4 is roughly 25–40 min per run. This is an estimate scaled from the RTX 4090, not a measurement.
The script uses fp16 autocast (not bf16), which the T4 supports. Expect about ±1 point of run-to-run variation
from GPU non-determinism.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/NadhaIrshad/EN4554-Deep-Learning-for-Vision-A01.git
%cd EN4554-Deep-Learning-for-Vision-A01/part1
!pip -q install "transformers>=4.46,<5" peft datasets accelerate num2words

## 1. Smoke test (about 5 min): checks the full pipeline on 200 examples

In [ ]:
!python -u part1_loraplus_smolvlm.py --quick --experiment single --lr-a 2e-4 --ratio 1 --seed 42

## 2. Tuned LoRA, full budget, seed 42

In [ ]:
!python -u part1_loraplus_smolvlm.py --experiment single --lr-a 2e-4 --ratio 1 --seed 42 --out-dir /content/repro

## 3. Tuned LoRA+ (λ = 4), full budget, seed 42

In [ ]:
!python -u part1_loraplus_smolvlm.py --experiment single --lr-a 1e-4 --ratio 4 --seed 42 --out-dir /content/repro

## 4. Compare with the committed results

The numbers printed above should be close to the committed `results.json` files (test accuracy, exact match on VQAv2).
To keep results after a disconnect, mount Drive and pass `--out-dir /content/drive/MyDrive/...`; finished runs are cached and skipped.

In [ ]:
import json, glob
for f in sorted(glob.glob("/content/repro/runs/single_*/results.json")):
    r = json.load(open(f)); print(f"{r['name']:32s} test acc {100*r['test_acc']:.2f}%  ({r['train_min']:.1f} min)")
for name in ["main_LoRA_tuned_lr0.0002_r1_s42", "main_LoRAplus_tuned_lr0.0001_r4_s42"]:
    r = json.load(open(f"outputs/part1_loraplus/runs/{name}/results.json")); print(f"committed {name:38s} test acc {100*r['test_acc']:.2f}%")